In [535]:
import us
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

In [536]:
customer_pop = pd.read_csv('data/city_pop.txt', sep='\t', skipinitialspace=True)
state_pop = pd.read_csv('data/state_pop.txt', sep='\t', skipinitialspace=True)
state_expenditure = pd.read_csv('data/weekly_retail_sales_per_state_and_category.txt', sep='\t', skipinitialspace=True)
customer_distances = pd.read_csv('data/city_distance.txt', sep='\t', skipinitialspace=True)
stores = pd.read_csv('data/store_preference_indices.txt', sep='\t', skipinitialspace=True)

In [537]:
states = us.states.STATES
states_abbr = {}
for state in states:
    key = state.name
    abbr = state.abbr
    states_abbr[key] = abbr

In [538]:
state_expenditure = state_expenditure.groupby(['State'], as_index=False).sum(numeric_only=True)

In [539]:
state_expenditure['State'] = state_expenditure['State'].replace(states_abbr)

In [540]:
state_pop['State'] = state_pop['State'].replace(states_abbr)

In [541]:
state_financing = state_pop.merge(state_expenditure, how='inner', on='State')

In [542]:
state_financing.head()

,State,estimated_pop_2022,Dollars
0,AL,5074296.0,1.498050e+10
1,AZ,7359197.0,2.007365e+10
2,AR,3045637.0,8.385400e+09
3,CA,39029342.0,8.108794e+10
4,CO,5839926.0,1.610612e+10


In [543]:
customer_expenditure = customer_pop.merge(state_financing, how='inner', left_on='state', right_on='State')

In [544]:
customer_expenditure['pop_fraction'] = customer_expenditure['population'] / customer_expenditure['estimated_pop_2022']

In [545]:
customer_expenditure['estimated_revenue'] = customer_expenditure['Dollars'] * customer_expenditure['pop_fraction']

In [546]:
customer_expenditure['city'] = customer_expenditure['city'] + ", " + customer_expenditure['state']

In [547]:
customer_expenditure.drop('state', inplace=True, axis=1)

In [548]:
customer_distances.head()

,city,"Youngstown, OH","Yankton, SD","Yakima, WA","Worcester, MA","Wisconsin Dells, WI","Winston-Salem, NC","Winnipeg, MB","Winchester, VA","Wilmington, NC",...,"Roanoke, VA","Richmond, VA","Richmond, IN","Richfield, UT","Rhinelander, WI","Reno, NV","Regina, SA","Red Bluff, CA","Reading, PA","Ravenna, OH"
0,"Youngstown, OH",0,966,1513,2964,1149,927,1611,1510,390,...,1018,168,565,1700,1636,2019,1458,1564,2871,348
1,"Yankton, SD",966,0,2410,1520,1817,729,686,290,1823,...,548,1127,432,2265,558,571,943,198,1917,2541
2,"Yakima, WA",1513,2410,0,604,481,2742,1833,826,214,...,752,486,590,2132,1095,2154,1211,2217,2673,1570
3,"Worcester, MA",2964,1520,604,0,595,1289,1446,466,1139,...,1861,861,499,1408,986,2719,1437,769,1038,2343
4,"Wisconsin Dells, WI",1149,1817,481,595,0,494,550,2641,765,...,216,1994,324,2187,260,2586,1974,2352,2243,691


In [549]:
city1, city2, distance = ([] for _ in range(3))

In [550]:
for i in customer_distances.index:
    city = customer_distances.loc[i, 'city']
    for column in customer_distances.columns[1:]:
        distance_n = customer_distances.loc[i, column]
        if distance_n == 0: break
        else:
            city1.append(city)
            city2.append(column)
            distance.append(distance_n)

In [551]:
df = pd.DataFrame({'city_a':city1, 'city_b':city2, 'distance':distance})

In [552]:
stores['city'] = stores['city'] + ', ' + stores['state']

In [553]:
stores.drop('state', axis=1, inplace=True)

In [554]:
stores.head()

,city,metro_traffic_rank,num_highways,max_m2_size_000,estimated_num_parking_space_0,rough_estimate_accessibility,rough_estimate_merchandizing,num_cbd
0,"Raleigh, NC",4.17,12,124.78,100,5,5,1
1,"Richmond, VA",4.50,27,94.68,100,6,6,1
2,"Riverside, CA",4.33,3,110.00,200,7,7,1
3,"Rochester, NY",3.17,8,151.43,100,3,3,1
4,"Sacramento, CA",3.50,6,103.01,200,7,7,1


In [555]:
proposed_stores = stores[stores['city'].isin(customer_distances['city'])]

In [556]:
proposed_stores.set_index('city', inplace=True)
customer_distances.set_index('city', inplace=True)
customer_expenditure.set_index('city', inplace=True)

In [ ]:
customer_expenditure = customer_expenditure['estimated_revenue']

In [502]:
proposed_stores_list = proposed_stores.index.unique()

In [503]:
selected_distances = [col for col in customer_distances.columns if col in proposed_stores_list]

In [504]:
customer_distances = customer_distances.loc[:,selected_distances]

In [505]:
customer_distances.replace(0,1,inplace=True)

In [506]:
scaler = MinMaxScaler()
X = scaler.fit_transform(proposed_stores)
X = pd.DataFrame(X, index=proposed_stores.index, columns=proposed_stores.columns)

In [507]:
X_sum = X.sum(axis=1)

In [508]:
coefficient = {}
for store in X_sum.index:
    coefficient[store] = X_sum[store]

In [509]:
store_pref, prob_df, expense_df = (pd.DataFrame([],index=customer_distances.index) for _ in range(3))

In [510]:
for col in customer_distances.columns:
    store_pref[col] = coefficient[col]/(customer_distances[col])**2

In [512]:
pref_sum = store_pref.sum(axis=1)

In [513]:
for col in store_pref.columns:
    prob_df[col] = store_pref[col]/pref_sum

In [515]:
for col in prob_df.columns:
    expense_df[col] = prob_df[col]*customer_expenditure

In [517]:
expense = expense_df.sum(axis=0).sort_values(ascending=False).to_dict()

In [518]:
preferred_store = [key for key in expense.keys() if expense[key] == max(expense.values())]

In [519]:
f'The preferred store location is {preferred_store[0]}'

'The preferred store location is San Antonio, TX'